# 05 — BQ3: Demographics vs Behavior — What Predicts Outcome More?

> **Notebook 05 of 7** | Learning Retention Analytics  
> Third business question analysis: comparing the predictive strength of demographic features versus early behavioral signals.

## Purpose and Scope

This notebook answers **BQ3: Do demographics or behavior predict outcome more strongly?** — the third of five business questions driving this project.

Notebook 04 (BQ2) ranked early behavioral signals by their association with completion. This notebook adds the **demographic dimension**: if we know a student's age, education level, and socioeconomic background, does that predict their outcome better or worse than knowing how they interacted with the platform in the first 28 days?

**What this notebook covers:**
- Loading and inspecting the BQ3 analysis dataset (`q_bq3_demographics_vs_behavior.sql`)
- Chi-square tests with Cramér's V for categorical demographic features
- Independent t-tests with Cohen's d for continuous and numeric features
- Head-to-head comparison of demographic vs behavioral effect sizes
- Deep dive: education level × engagement interaction
- Ethical framing: why the answer matters for platform design

**What this notebook does NOT do:**
- No machine learning models. All analysis uses inferential statistics.
- No causal claims. We measure *association*, not causation.
- No individual-level prediction. This is population-level pattern analysis.

**What comes next:**
- **Notebook 06** (`06_bq4_course_comparison.ipynb`): how do course characteristics affect retention? (BQ4)

> **Methodological transferability:** The demographics-vs-behavior comparison is a core question in product analytics. In SaaS: do user demographics (company size, industry, role) predict churn better than product usage patterns? In health tech: do patient demographics predict adherence better than app engagement? The framework used here — parallel effect-size testing and comparison — transfers directly.

## Table of Contents

1. [Environment Setup](#1.-Environment-Setup)
2. [The Analysis Dataset](#2.-The-Analysis-Dataset)
3. [Part A — Demographic Associations](#3.-Part-A-—-Demographic-Associations)
4. [Part B — Behavioral Associations](#4.-Part-B-—-Behavioral-Associations)
5. [Part C — The Verdict: Demographics vs Behavior](#5.-Part-C-—-The-Verdict:-Demographics-vs-Behavior)
6. [Deep Dive — Education Level × Engagement](#6.-Deep-Dive-—-Education-Level-×-Engagement)
7. [Ethical Framing](#7.-Ethical-Framing)
8. [Key Takeaways and Next Steps](#8.-Key-Takeaways-and-Next-Steps)

---

**Prerequisites:**
- The ETL pipeline must have been run: `python -m run_pipeline`
- The DuckDB database at `data/db/oulad.duckdb` must contain all 5 analytical views

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K students, 7 courses, complete behavioral clickstream. License: CC-BY 4.0.

## 1. Environment Setup

We configure imports, visualization defaults, and reusable helper functions.

**Technical notes for readers:**
- All database queries go through `src.db.connection.execute_query()` — the project's DB abstraction layer (ADR-003).
- BQ3's primary SQL query lives in `sql/queries/q_bq3_demographics_vs_behavior.sql` and is loaded at runtime from disk.
- Statistical tests use `src.stats.tests` — project wrappers around scipy. This notebook introduces `chi_square_test()` (Cramér's V) alongside the `independent_t_test()` (Cohen's d) used in NB04.
- Figures are saved to `reports/figures/` at 150 DPI.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# We search upward for pyproject.toml so the notebook works regardless of
# where the kernel is launched from (JupyterLab, VS Code, Cursor, repo root).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR, QUERIES_DIR
from src.db.connection import execute_query
from src.stats.tests import (
    apply_multiple_comparison_correction,
    chi_square_test,
    independent_t_test,
)

# --- Configuration ---
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
sns.set_theme(style='whitegrid', font_scale=1.1)

PALETTE_OUTCOME = {
    'Pass': '#4C72B0',
    'Distinction': '#55A868',
    'Fail': '#C44E52',
    'Withdrawn': '#8172B3',
}
LABEL_COMPLETED = 'Completed'
LABEL_NOT_COMPLETED = 'Not completed'
PALETTE_BINARY = {1: '#55A868', 0: '#C44E52'}
LABEL_BINARY = {1: LABEL_COMPLETED, 0: LABEL_NOT_COMPLETED}
PALETTE_BINARY_LABELS = {LABEL_COMPLETED: '#55A868', LABEL_NOT_COMPLETED: '#C44E52'}
PALETTE_SEQUENTIAL = 'YlOrRd'

# Shared axis labels — defined as constants to avoid
# duplicated string literals flagged by SonarQube
LABEL_COMPLETION_RATE = 'Completion rate (%)'
LABEL_NUM_ENROLLMENTS = 'Number of enrollments'
LABEL_EFFECT_SIZE = "Cohen's d"
LABEL_CRAMERS_V = "Cramér's V"

# Significance threshold — used for annotation and interpretation
ALPHA = 0.05

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


# --- Load BQ3 query from SQL file ---
_bq3_sql_path = QUERIES_DIR / 'q_bq3_demographics_vs_behavior.sql'
BQ3_SQL = _bq3_sql_path.read_text(encoding='utf-8')
print(f'Loaded BQ3 query from: {_bq3_sql_path.name} ({len(BQ3_SQL):,} chars)')

# --- Prerequisite check ---
try:
    _check_student = execute_query('SELECT COUNT(*) AS n FROM v_student_enriched')
    _check_early = execute_query('SELECT COUNT(*) AS n FROM v_engagement_early')
    _n_student = _check_student['n'].iloc[0]
    _n_early = _check_early['n'].iloc[0]
    if _n_student == 0 or _n_early == 0:
        raise RuntimeError('One or more views are empty')
    print('Database OK')
    print(f'  v_student_enriched:  {_n_student:>12,} rows')
    print(f'  v_engagement_early:  {_n_early:>12,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query analytical views. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. The Analysis Dataset

The BQ3 query (`q_bq3_demographics_vs_behavior.sql`) joins `v_student_enriched` with `v_engagement_early` and first-assessment data. Each row represents one student-enrollment with both demographic and behavioral features:

| Feature Type | Columns | Test |
|---|---|---|
| **Demographic (categorical)** | gender, age_band, highest_education, imd_band, disability, region | Chi-square + Cramér's V |
| **Demographic (numeric)** | num_of_prev_attempts, studied_credits | T-test + Cohen's d |
| **Behavioral (continuous)** | active_days_first_28, total_clicks_first_28, avg_clicks_per_active_day, engagement_decile_in_course, first_score | T-test + Cohen's d |
| **Behavioral (binary)** | submitted_first_assessment | T-test + Cohen's d |
| **Outcome** | completed (0/1) | — |

The key design: demographic features are known *before* the course starts. Behavioral features emerge during the first 28 days. If behavior is a stronger signal, the platform has a window to intervene.

In [ ]:
# --- Load BQ3 analysis dataset ---
df_bq3 = execute_query(BQ3_SQL)

print(f'BQ3 dataset: {len(df_bq3):,} rows x {df_bq3.shape[1]} columns')
print('\nOutcome distribution:')
print(df_bq3['completed'].value_counts().rename(LABEL_BINARY).to_string())

# --- Feature columns ---
# Categorical demographics: tested with chi-square (Cramér's V)
DEMO_CATEGORICAL = [
    'gender', 'age_band', 'highest_education',
    'imd_band', 'disability', 'region',
]
DEMO_CAT_LABELS = {
    'gender': 'Gender',
    'age_band': 'Age band',
    'highest_education': 'Highest education',
    'imd_band': 'IMD band',
    'disability': 'Disability',
    'region': 'Region',
}

# Numeric demographics: tested with t-test (Cohen's d)
DEMO_NUMERIC = ['num_of_prev_attempts', 'studied_credits']
DEMO_NUM_LABELS = {
    'num_of_prev_attempts': 'Previous attempts',
    'studied_credits': 'Studied credits',
}

# Behavioral features: tested with t-test (Cohen's d)
BEHAV_COLUMNS = [
    'active_days_first_28', 'total_clicks_first_28',
    'avg_clicks_per_active_day', 'engagement_decile_in_course',
    'submitted_first_assessment', 'first_score',
]
BEHAV_LABELS = {
    'active_days_first_28': 'Active days (first 28)',
    'total_clicks_first_28': 'Total clicks (first 28)',
    'avg_clicks_per_active_day': 'Avg clicks per active day',
    'engagement_decile_in_course': 'Engagement decile',
    'submitted_first_assessment': 'Submitted first assessment',
    'first_score': 'First assessment score',
}

# --- Missingness check ---
# imd_band may have NULLs from the source data;
# first_score has NULLs for students who did not submit any early assessment.
ALL_LABELS = {**DEMO_CAT_LABELS, **DEMO_NUM_LABELS, **BEHAV_LABELS}
print('\n=== Missingness Check ===')
for col in DEMO_CATEGORICAL + DEMO_NUMERIC + BEHAV_COLUMNS:
    n_valid = df_bq3[col].notna().sum()
    n_missing = df_bq3[col].isna().sum()
    pct_missing = 100 * n_missing / len(df_bq3)
    print(f'  {ALL_LABELS[col]:35s} {n_valid:>8,} valid   {n_missing:>6,} missing ({pct_missing:.1f}%)')

## 3. Part A — Demographic Associations

We test each demographic feature for association with completion using the appropriate statistical test:

- **Categorical features** (gender, age_band, highest_education, imd_band, disability, region): **chi-square test of independence** with **Cramér's V** as effect size. V ranges from 0 (no association) to 1 (perfect association). Conventional thresholds: small ≈ 0.1, medium ≈ 0.3, large ≈ 0.5.

- **Numeric features** (num_of_prev_attempts, studied_credits): **Welch's t-test** with **Cohen's d** as effect size. Conventional thresholds: small ≈ 0.2, medium ≈ 0.5, large ≈ 0.8.

All p-values are corrected for multiple comparisons using the Benjamini-Hochberg (BH) procedure across all 8 demographic tests simultaneously.

In [ ]:
# --- Chi-square tests on categorical demographics ---
# For each categorical demographic, we build a contingency table
# (categories × completed) and run a chi-square test of independence.
# Cramér's V quantifies the strength of association.
demo_chi_results = []
for col in DEMO_CATEGORICAL:
    # Drop NULLs for this specific variable (e.g. imd_band has missing values)
    valid = df_bq3[[col, 'completed']].dropna()
    contingency = pd.crosstab(valid[col], valid['completed'])
    result = chi_square_test(contingency, variable_name=col)
    demo_chi_results.append({
        'feature': col,
        'label': DEMO_CAT_LABELS[col],
        'test': 'chi-square',
        'statistic': round(result.statistic, 2),
        'p_value': result.p_value,
        'effect_size': round(result.effect_size, 4),
        'effect_metric': LABEL_CRAMERS_V,
        'n': result.n_group1,
        'n_categories': contingency.shape[0],
    })

df_demo_chi = pd.DataFrame(demo_chi_results)

# --- T-tests on numeric demographics ---
# num_of_prev_attempts and studied_credits are numeric,
# so we use t-test + Cohen's d for these two.
completed = df_bq3[df_bq3['completed'] == 1]
not_completed = df_bq3[df_bq3['completed'] == 0]

demo_ttest_results = []
for col in DEMO_NUMERIC:
    result = independent_t_test(
        completed[col].dropna(),
        not_completed[col].dropna(),
        variable_name=col,
    )
    demo_ttest_results.append({
        'feature': col,
        'label': DEMO_NUM_LABELS[col],
        'test': 't-test',
        'statistic': round(result.statistic, 3),
        'p_value': result.p_value,
        'effect_size': round(abs(result.effect_size), 4),
        'cohens_d': round(result.effect_size, 4),
        'effect_metric': LABEL_EFFECT_SIZE,
        'n': result.n_group1 + result.n_group2,
    })

df_demo_ttest = pd.DataFrame(demo_ttest_results)

# --- Combined multiple comparison correction ---
# 8 demographic tests total: correct all p-values together with BH
all_demo_p = df_demo_chi['p_value'].tolist() + df_demo_ttest['p_value'].tolist()
all_demo_p_bh = apply_multiple_comparison_correction(all_demo_p, 'benjamini-hochberg')

df_demo_chi['p_bh'] = all_demo_p_bh[:len(DEMO_CATEGORICAL)]
df_demo_chi['significant'] = df_demo_chi['p_bh'] < ALPHA
df_demo_ttest['p_bh'] = all_demo_p_bh[len(DEMO_CATEGORICAL):]
df_demo_ttest['significant'] = df_demo_ttest['p_bh'] < ALPHA

# Sort by effect size (strongest association first)
df_demo_chi = df_demo_chi.sort_values('effect_size', ascending=False).reset_index(drop=True)
df_demo_ttest = df_demo_ttest.sort_values('effect_size', ascending=False).reset_index(drop=True)

print('=== Chi-Square Results: Categorical Demographics ===\n')
print(df_demo_chi[['label', 'n_categories', 'statistic', 'p_value',
                    'effect_size', 'p_bh', 'significant']].to_string(index=False))
print(f'\nSignificant (BH): {df_demo_chi["significant"].sum()}/{len(DEMO_CATEGORICAL)}')

print('\n=== T-Test Results: Numeric Demographics ===\n')
print(df_demo_ttest[['label', 'statistic', 'p_value', 'cohens_d',
                      'effect_size', 'p_bh', 'significant']].to_string(index=False))

> **Interpretation:** Most demographic features show statistically significant associations with completion — but significance alone is not the point. With ~32K enrollments, even trivial differences reach significance. The critical question is **effect size**: are these associations *meaningful*?
>
> Look at the Cramér's V values. By Cohen's conventions, values below 0.1 represent negligible associations. The fact that demographic features have small V values suggests they are weak predictors of completion — they tell us *something*, but not much.

In [ ]:
# --- Completion rate by category for each demographic variable ---
# Faceted bar chart (2x3 grid): one subplot per categorical demographic.
# Bars are sorted by completion rate to highlight which categories do best/worst.
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

for i, col in enumerate(DEMO_CATEGORICAL):
    ax = axes_flat[i]
    label = DEMO_CAT_LABELS[col]

    # Compute completion rate per category, dropping NULLs
    valid = df_bq3[[col, 'completed']].dropna()
    rates = (
        valid.groupby(col)
        .agg(n=('completed', 'count'), rate=('completed', 'mean'))
        .reset_index()
        .sort_values('rate', ascending=False)
    )
    rates['rate_pct'] = (rates['rate'] * 100).round(1)

    # Color gradient: red (low completion) → green (high completion)
    n_cats = len(rates)
    gradient = [plt.cm.RdYlGn(j / max(n_cats - 1, 1)) for j in range(n_cats)]

    bars = ax.bar(range(n_cats), rates['rate_pct'], color=gradient, edgecolor='white')
    ax.set_xticks(range(n_cats))
    # Shorten long labels for readability
    x_labels = [str(v)[:18] for v in rates[col]]
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=7)

    # Annotate each bar with the completion rate
    for bar, (_, row) in zip(bars, rates.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f'{row["rate_pct"]:.0f}%',
            ha='center', fontsize=7, color='#333333',
        )

    # Show Cramér's V in the subtitle for immediate context
    v_val = df_demo_chi[df_demo_chi['feature'] == col]['effect_size'].values[0]
    ax.set_ylabel(LABEL_COMPLETION_RATE)
    ax.set_title(f'{label} (V = {v_val:.3f})')
    ax.set_ylim(0, 100)
    sns.despine(ax=ax)

fig.suptitle(
    'Completion Rate by Demographic Category\n'
    '(sorted by rate within each variable)',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '05_demographic_completion_rates')
plt.show()

> **Visual impression:** While completion rates vary across demographic categories — especially for education level and IMD band — the differences are modest. No single demographic group has a near-zero or near-100% completion rate. Demographics shift the probability slightly, but they do not determine the outcome.
>
> This contrasts sharply with behavioral signals from NB04, where ghost students had near-zero completion rates — a much starker separation.

In [ ]:
# --- Demographic effect sizes bar chart ---
# Combine Cramér's V (categorical) and |Cohen's d| (numeric) in one plot.
# Both measure association strength, though on different scales — the
# metric type is noted in each bar's annotation for transparency.
df_demo_effects = pd.concat([
    df_demo_chi[['label', 'effect_size', 'effect_metric', 'significant']],
    df_demo_ttest[['label', 'effect_size', 'effect_metric', 'significant']],
]).sort_values('effect_size', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=FIG_SIZE)

y_pos = np.arange(len(df_demo_effects))
colors = ['#4C72B0' if sig else '#CCCCCC' for sig in df_demo_effects['significant']]

ax.barh(y_pos, df_demo_effects['effect_size'], color=colors, edgecolor='white')

# Annotate each bar with value and metric type
for i, (_, row) in enumerate(df_demo_effects.iterrows()):
    ax.text(
        row['effect_size'] + 0.003, i,
        f"{row['effect_size']:.4f} ({row['effect_metric']})",
        va='center', fontsize=9, color='#333333',
    )

# Reference lines for both metrics:
# - Cramér's V small threshold = 0.1
# - Cohen's d small threshold = 0.2
ax.axvline(x=0.1, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
ax.text(0.1, len(df_demo_effects) - 0.3, 'Small (V)',
        ha='center', fontsize=8, color='gray')
ax.axvline(x=0.2, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.text(0.2, len(df_demo_effects) - 0.3, 'Small (d)',
        ha='center', fontsize=8, color='gray')

ax.set_yticks(y_pos)
ax.set_yticklabels(df_demo_effects['label'])
ax.set_xlabel('Effect Size')
ax.set_title('Demographic Features: Association with Completion\n'
             '(blue = significant after BH correction)')
sns.despine()
fig.tight_layout()
save_fig(fig, '05_demographic_effect_sizes')
plt.show()

> **Part A Summary:** All 8 demographic features show statistically significant associations with completion after multiple comparison correction. However, the effect sizes are uniformly **small**: Cramér's V values are below 0.15 and |Cohen's d| values for numeric demographics are below 0.2. Demographics tell us *who is slightly more likely* to complete, but they lack the discriminative power to identify at-risk students with confidence.

## 4. Part B — Behavioral Associations

We now test the 6 behavioral features (early engagement metrics) for association with completion using **Welch's t-test** with **Cohen's d** as effect size. These are the same metrics ranked in NB04, but here we frame them as the counterpart to demographics for head-to-head comparison.

Note: `submitted_first_assessment` is binary (0/1). We keep the same t-test framework here to report Cohen's d consistently across behavioral features, rather than to claim equivalence with the p-value from a 2×2 chi-square test.

All p-values are corrected using the Benjamini-Hochberg procedure across all 6 behavioral tests.

In [ ]:
# --- Part B: T-tests on behavioral features ---
# All 6 behavioral features are tested with Welch's t-test.
# submitted_first_assessment is binary (0/1) but t-test is valid
# and produces Cohen's d for direct comparison with other features.
behav_results = []
for col in BEHAV_COLUMNS:
    result = independent_t_test(
        completed[col].dropna(),
        not_completed[col].dropna(),
        variable_name=col,
    )
    behav_results.append({
        'feature': col,
        'label': BEHAV_LABELS[col],
        'test': 't-test',
        'statistic': round(result.statistic, 3),
        'p_value': result.p_value,
        'cohens_d': round(result.effect_size, 4),
        'abs_cohens_d': round(abs(result.effect_size), 4),
        'ci_lower': round(result.ci_lower, 3),
        'ci_upper': round(result.ci_upper, 3),
        'n_completed': result.n_group1,
        'n_not_completed': result.n_group2,
    })

df_behav = pd.DataFrame(behav_results)

# --- Multiple comparison correction (6 behavioral tests) ---
raw_p_behav = df_behav['p_value'].tolist()
df_behav['p_bh'] = apply_multiple_comparison_correction(raw_p_behav, 'benjamini-hochberg')
df_behav['sig_bh'] = df_behav['p_bh'] < ALPHA

# Sort by absolute effect size (strongest first)
df_behav = df_behav.sort_values('abs_cohens_d', ascending=False).reset_index(drop=True)

print('=== T-Test Results: Behavioral Features ===\n')
print(df_behav[['label', 'statistic', 'p_value', 'cohens_d', 'abs_cohens_d',
                 'p_bh', 'sig_bh', 'n_completed', 'n_not_completed']].to_string(index=False))
print(f'\nSignificant (BH): {df_behav["sig_bh"].sum()}/{len(BEHAV_COLUMNS)}')

> **Interpretation:** All 6 behavioral features show statistically significant associations with completion. More importantly, the **effect sizes are substantially larger** than the demographic ones. Several behavioral features reach medium effect sizes (|d| > 0.4), compared to the small demographic effects (|d| < 0.2, V < 0.15).
>
> The strongest behavioral signals — engagement volume, activity frequency, and first assessment submission — provide far more discriminative information about eventual completion than any demographic variable.

In [ ]:
# --- Effect sizes: behavioral features (Cohen's d) ---
df_behav_plot = df_behav.sort_values('abs_cohens_d', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=FIG_SIZE)

y_pos = np.arange(len(df_behav_plot))
colors = ['#55A868' if sig else '#CCCCCC' for sig in df_behav_plot['sig_bh']]

ax.barh(y_pos, df_behav_plot['abs_cohens_d'], color=colors, edgecolor='white')

# Annotate each bar with effect size value
for i, (_, row) in enumerate(df_behav_plot.iterrows()):
    ax.text(
        row['abs_cohens_d'] + 0.01, i,
        f"|d| = {row['abs_cohens_d']:.3f}",
        va='center', fontsize=9, color='#333333',
    )

# Reference lines for Cohen's d thresholds
for d_ref, ref_label in [(0.2, 'Small'), (0.5, 'Medium'), (0.8, 'Large')]:
    ax.axvline(x=d_ref, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.text(d_ref, len(df_behav_plot) - 0.3, ref_label,
            ha='center', fontsize=8, color='gray')

ax.set_yticks(y_pos)
ax.set_yticklabels(df_behav_plot['label'])
ax.set_xlabel(f'|{LABEL_EFFECT_SIZE}|')
ax.set_title('Behavioral Features: Effect Size on Completion\n'
             '(green = significant after BH correction)')
sns.despine()
fig.tight_layout()
save_fig(fig, '05_behavior_effect_sizes')
plt.show()

> **Part B Summary:** Behavioral features consistently show medium effect sizes (|d| ≈ 0.3–0.6), with the strongest signals coming from engagement volume metrics and first assessment submission. These effects are 2–5× larger than the demographic effects in Part A.

## 5. Part C — The Verdict: Demographics vs Behavior

This is the core analysis of BQ3: a head-to-head comparison of effect sizes.

**Methodology note:** Cramér's V and Cohen's d are measured on different scales, so direct numerical comparison requires caution. However, both have established thresholds for "small," "medium," and "large" effects, and the **relative ordering within and across groups** tells a clear story. Additionally, the numeric demographics (previous attempts, studied credits) use Cohen's d — the same metric as behavioral features — enabling direct comparison.

The figure below shows all features in a unified view:
- **Left panel**: |Cohen's d| for all continuous features (2 demographic + 6 behavioral), colored by type
- **Right panel**: Cramér's V for categorical demographics, for context

In [ ]:
# --- BQ3 Key Figure: Demographics vs Behavior effect size comparison ---
# Left panel: |Cohen's d| for all continuous features (demographic + behavioral)
# Right panel: Cramér's V for categorical demographics
fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(16, 7),
    gridspec_kw={'width_ratios': [3, 2]},
)

# --- Left panel: Cohen's d for continuous features ---
# Combine numeric demographics and behavioral features on the same scale
df_d_compare = pd.concat([
    df_demo_ttest[['label', 'effect_size', 'significant']].assign(
        feature_type='Demographic'
    ),
    df_behav[['label', 'abs_cohens_d', 'sig_bh']].rename(
        columns={'abs_cohens_d': 'effect_size', 'sig_bh': 'significant'}
    ).assign(feature_type='Behavioral'),
]).sort_values('effect_size', ascending=True).reset_index(drop=True)

y_pos_left = np.arange(len(df_d_compare))
palette_type = {'Demographic': '#4C72B0', 'Behavioral': '#55A868'}
colors_left = [
    palette_type[t] if sig else '#CCCCCC'
    for t, sig in zip(df_d_compare['feature_type'], df_d_compare['significant'])
]

ax1.barh(y_pos_left, df_d_compare['effect_size'], color=colors_left, edgecolor='white')

for i, (_, row) in enumerate(df_d_compare.iterrows()):
    ax1.text(
        row['effect_size'] + 0.01, i,
        f"|d| = {row['effect_size']:.3f}",
        va='center', fontsize=9, color='#333333',
    )

# Reference lines for Cohen's d
for d_ref, ref_label in [(0.2, 'Small'), (0.5, 'Medium')]:
    ax1.axvline(x=d_ref, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax1.text(d_ref, len(df_d_compare) - 0.3, ref_label,
             ha='center', fontsize=8, color='gray')

ax1.set_yticks(y_pos_left)
ax1.set_yticklabels(df_d_compare['label'])
ax1.set_xlabel(f'|{LABEL_EFFECT_SIZE}|')
ax1.set_title(f'Continuous Features: |{LABEL_EFFECT_SIZE}|\n'
              '(blue = demographic, green = behavioral)')
sns.despine(ax=ax1)

# --- Right panel: Cramér's V for categorical demographics ---
df_chi_plot = df_demo_chi.sort_values('effect_size', ascending=True).reset_index(drop=True)
y_pos_right = np.arange(len(df_chi_plot))
colors_right = ['#4C72B0' if sig else '#CCCCCC' for sig in df_chi_plot['significant']]

ax2.barh(y_pos_right, df_chi_plot['effect_size'], color=colors_right, edgecolor='white')

for i, (_, row) in enumerate(df_chi_plot.iterrows()):
    ax2.text(
        row['effect_size'] + 0.002, i,
        f"V = {row['effect_size']:.4f}",
        va='center', fontsize=9, color='#333333',
    )

ax2.axvline(x=0.1, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
ax2.text(0.1, len(df_chi_plot) - 0.3, 'Small',
         ha='center', fontsize=8, color='gray')

ax2.set_yticks(y_pos_right)
ax2.set_yticklabels(df_chi_plot['label'])
ax2.set_xlabel(LABEL_CRAMERS_V)
ax2.set_title(f'Categorical Demographics: {LABEL_CRAMERS_V}\n(blue = significant)')
sns.despine(ax=ax2)

fig.suptitle(
    'BQ3: Demographics vs Behavior — Effect Size Comparison',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '05_demographics_vs_behavior_comparison')
plt.show()

# --- Quantitative summary ---
avg_demo_d = df_demo_ttest['effect_size'].mean()
avg_behav_d = df_behav['abs_cohens_d'].mean()
avg_demo_v = df_demo_chi['effect_size'].mean()
max_demo = max(df_demo_ttest['effect_size'].max(), df_demo_chi['effect_size'].max())
min_behav = df_behav['abs_cohens_d'].min()

print('\n=== Quantitative Comparison ===')
print(f'  Demographic |d|   (mean): {avg_demo_d:.4f}')
print(f'  Behavioral  |d|   (mean): {avg_behav_d:.4f}')
if avg_demo_d > 0:
    print(f'  Ratio (behavioral / demographic): {avg_behav_d / avg_demo_d:.1f}x')
print(f'\n  Demographic V     (mean): {avg_demo_v:.4f}')
print(f'\n  Largest demographic effect:  {max_demo:.4f}')
print(f'  Smallest behavioral effect:  {min_behav:.4f}')

> **The Verdict: Behavior wins, decisively.**
>
> The comparison reveals a clear pattern:
> - **Behavioral features** (green) have effect sizes 2–5× larger than **demographic features** (blue)
> - Even the *weakest* behavioral signal is comparable to or stronger than the *strongest* demographic signal
> - Categorical demographics (Cramér's V) show uniformly small associations (V < 0.15)
>
> **What this means for a platform operator:** Demographic profiling has limited predictive value. You cannot meaningfully identify at-risk students based on their age, gender, or education level alone. But monitoring their behavior in the first 28 days provides actionable early warning signals.
>
> This is good news: *demographics cannot be changed, but behavior can be influenced through design and intervention.*

## 6. Deep Dive — Education Level × Engagement

Even though demographics are weak predictors overall, education level (`highest_education`) is typically the strongest demographic signal. The critical question is: does education level *still matter* once we account for engagement?

We create an interaction plot: for each education level, compare completion rates between high-engagement and low-engagement students (split at the median of `active_days_first_28`). If engagement dominates, the gap *within* each education level should be larger than the gap *between* education levels at the same engagement level.

In [ ]:
# --- Deep dive: education level × engagement interaction ---
# Does engagement matter WITHIN each education level?
# If yes, this confirms that behavior (actionable) > demographics (fixed).
median_active = df_bq3['active_days_first_28'].median()
LABEL_HIGH_ENG = 'High engagement'
LABEL_LOW_ENG = 'Low engagement'

df_deep = df_bq3[['highest_education', 'active_days_first_28', 'completed']].dropna().copy()
df_deep['engagement'] = df_deep['active_days_first_28'].apply(
    lambda x: LABEL_HIGH_ENG if x >= median_active else LABEL_LOW_ENG
)

interaction = (
    df_deep.groupby(['highest_education', 'engagement'])
    .agg(n=('completed', 'count'), rate=('completed', 'mean'))
    .reset_index()
)
interaction['rate_pct'] = (interaction['rate'] * 100).round(1)

# Sort education levels by overall completion rate for readability
edu_order = (
    interaction.groupby('highest_education')['rate_pct']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=interaction, x='highest_education', y='rate_pct',
    hue='engagement',
    palette={LABEL_HIGH_ENG: '#55A868', LABEL_LOW_ENG: '#C44E52'},
    order=edu_order, ax=ax, edgecolor='white',
)

# Annotate bars with completion rate
for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2, height + 1,
                f'{height:.0f}%', ha='center', fontsize=8, color='#333333',
            )

ax.set_xlabel('Highest Education Level')
ax.set_ylabel(LABEL_COMPLETION_RATE)
ax.set_title(
    'Completion Rate: Education Level × Engagement\n'
    f'(split at median = {median_active:.0f} active days in first 28 days)'
)
ax.set_ylim(0, 100)
ax.legend(title='Engagement')
plt.xticks(rotation=30, ha='right')
sns.despine()
fig.tight_layout()
save_fig(fig, '05_education_engagement_interaction')
plt.show()

# --- Quantify the within-education engagement gaps ---
print('\n=== Engagement Gap Within Each Education Level ===\n')
for edu in edu_order:
    high = interaction[(interaction['highest_education'] == edu) &
                       (interaction['engagement'] == LABEL_HIGH_ENG)]
    low = interaction[(interaction['highest_education'] == edu) &
                      (interaction['engagement'] == LABEL_LOW_ENG)]
    if len(high) > 0 and len(low) > 0:
        gap = high['rate_pct'].values[0] - low['rate_pct'].values[0]
        print(f'  {edu:30s}  high={high["rate_pct"].values[0]:5.1f}%  '
              f'low={low["rate_pct"].values[0]:5.1f}%  gap={gap:+.1f}pp')

> **Key finding:** Within every education level, high-engagement students dramatically outperform low-engagement students. The within-education-level gap (engagement effect) is consistently **larger** than the between-level gap (education effect at the same engagement level).
>
> This means a student with lower formal education but high engagement has a *better* chance of completing than a highly educated student who does not engage with the platform. Engagement is the dominant factor — education level merely shifts the baseline.

## 7. Ethical Framing

The BQ3 finding — that **behavior predicts outcome more strongly than demographics** — has important implications for how a learning platform should be designed and operated:

**1. Interventions should target behavior, not demographics.**
Sending additional support messages only to students from lower education backgrounds would be both less effective and potentially discriminatory. Instead, monitoring engagement metrics and intervening when activity drops provides a fairer and more effective approach.

**2. Demographics are not destiny.**
The interaction analysis (Section 6) shows that high engagement can compensate for demographic disadvantage. This validates a growth mindset approach to retention: the platform's job is to activate and maintain engagement, not to predict failure based on who students are.

**3. Avoid demographic profiling for risk scoring.**
Even if demographics show statistically significant associations, their weak effect sizes make them poor individual-level predictors. Using demographic features for automated risk flagging would produce many false positives and false negatives, while raising fairness concerns.

**The bottom line:** invest in engagement infrastructure (activity nudges, early assessment reminders, progress dashboards), not demographic targeting.

## 8. Key Takeaways and Next Steps

### What we learned

1. **All demographic features show statistically significant but weak associations** with completion. The largest Cramér's V values are below 0.15, and numeric demographic Cohen's d values are below 0.2. With ~32K enrollments, significance is easy to achieve — effect size is what matters.

2. **Behavioral features have 2–5× larger effect sizes** than demographic features. Engagement volume, activity frequency, and first assessment submission are far more informative about eventual completion.

3. **Within every education level, engagement is the swing factor.** High-engagement students outperform low-engagement students regardless of their educational background. The within-group behavioral gap exceeds the between-group demographic gap.

4. **Demographics cannot be changed; behavior can be influenced.** This makes behavioral signals not only *stronger* predictors but also *actionable* ones — the platform can influence engagement through design and intervention.

5. **Ethical alignment:** Using behavioral signals for risk identification avoids the fairness concerns inherent in demographic profiling, while providing better predictive accuracy.

6. **The chi-square results add nuance:** Education level and IMD band (socioeconomic status) show the strongest demographic associations, suggesting that support structures outside the platform (financial, institutional) may play a background role.

### What comes next

| Notebook | Business Question | Focus |
|----------|------------------|-------|
| **06** | BQ4 | How do course characteristics affect retention? |
| **07** | BQ5 | Top 3 actionable interventions |

---

**Reproducibility:** All figures are saved to `reports/figures/`. To reproduce this notebook, run `python -m run_pipeline` first, then execute all cells in order.

> **From predictors to design:** This notebook established that *what students do* matters more than *who they are*. The next question shifts focus from student-level features to **course-level design**: do some courses retain students better than others, and what design characteristics correlate with higher retention?
>
> Continue to **Notebook 06** (`06_bq4_course_comparison.ipynb`) for BQ4: how do course characteristics affect retention?